# 参照实验：官方 DiffAE (Bedroom128) 的 D(t) 测试

**目的**：你们的模型测出 D(t)≈0（z_sem 通路死亡）。这个 notebook 对**官方 DiffAE 预训练模型**
跑完全相同的测试，作为健康参照：
- 如果 DiffAE 的 D(t) **稳定为正、有 t 结构、且 D(mismatch) 与 D(zero) 同涨**（信息被真正使用的签名）
  → 测试方法学成立，"通路死亡"是你们训练设定特有的问题；
- 如果 DiffAE 也是零线 → 要么是普遍现象（大新闻），要么方法有系统性盲区，需要重审。

**判据（信息被使用的唯一有效签名）**：D(mismatch) 在相同 t 处与 D(zero) 一起显著为正
（错误信息至少和不给信息一样糟）。只看 D(zero) 会被 OOD 扰动伪影骗到：
z_sem 若 98% 是常量，"置零"是 35 倍于"错位"的扰动，鼓包只反映扰动幅度。

**DiffAE 与你们模型的关键差异**：DiffAE 在**全部 1000 个 t** 上训练（无 spaced/custom 限制），
所以本测试全扫 t=0..999；z_sem 为 512 维（style_ch=512，与你们 cond_dim 一致）；
cond 注入方式由源码自动探测（`cond=` 或 `y=` 参数）。

**数据**：优先 LSUN bedroom val；没有就先用 ADE20K val（大量室内场景，域略有偏移，
不影响"cond 是否被使用"的判定）。

**运行**：依次 Run All。首次运行会自动 clone 仓库、装依赖、从 Google Drive 下载 checkpoint
（bedroom128_autoenc 的 last.ckpt，约几百 MB）。


In [ ]:
# ============ 路径与参数 ============
DIFFAE_ROOT = '/root/diffae'
CKPT_DIR    = DIFFAE_ROOT + '/checkpoints/bedroom128_autoenc'
CKPT_PATH   = CKPT_DIR + '/last.ckpt'
# 优先换 LSUN bedroom val；没有就用 ADE20K（与你们实验同源，对比方便）
IMG_DIR   = '/root/diffusion_for_represent_learning/Segmentation/mmsegmentation/data/ade/ADEChallengeData2016/images/validation'
OUT_DIR   = CKPT_DIR + '/zsem_gap_test'

VAL_N         = 512
SAMPLES_PER_T = 128    # 每个 t 的独立 (图,噪声) 采样数；想更稳就加大
BATCH_PER_T   = 32     # 每次前向 batch（显存不够调小，需整除 SAMPLES_PER_T）
DEVICE        = 'cuda'
SEED          = 0

import os, sys, glob
import torch
import numpy as np
import pandas as pd
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device(DEVICE)

# ---- 获取 diffae 代码（subprocess 版，任何环境下行为一致）----
import subprocess
if not os.path.exists(os.path.join(DIFFAE_ROOT, 'config.py')):
    subprocess.run(['git', 'clone', 'https://github.com/phizaz/diffae.git', DIFFAE_ROOT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lmdb', 'gdown', 'blobfile'], check=True)
sys.path.insert(0, DIFFAE_ROOT)
print('diffae repo ready at', DIFFAE_ROOT)


In [ ]:
# ---- 下载 checkpoint（Google Drive 文件夹：last.ckpt + latent.ckpt）----
if not os.path.exists(CKPT_PATH):
    subprocess.run(['gdown', '--folder',
                    'https://drive.google.com/drive/folders/12EdjbIKnvP5RngKsR0UU-4kgpPAaYtlp',
                    '-O', CKPT_DIR], check=True)
assert os.path.exists(CKPT_PATH), '下载失败，手动从 drive 文件夹下载 last.ckpt 到 ' + CKPT_DIR
print('ckpt:', CKPT_PATH, f'{os.path.getsize(CKPT_PATH)/1e9:.2f} GB')


## 1. 构建模型并加载 EMA 权重

手动复现 `bedroom128_autoenc()` 的配置（不 import templates，避开 pytorch_lightning 依赖）。
checkpoint 是 PyTorch Lightning 格式：`state_dict` 里 `model.*` 是在线权重、`ema_model.*` 是 EMA 权重。


In [ ]:
from config import TrainConfig, ModelName, GenerativeType

conf = TrainConfig()
conf.batch_size = 32
conf.beatgans_gen_type = GenerativeType.ddim
conf.beta_scheduler = 'linear'
conf.data_name = 'bedroom256'
conf.diffusion_type = 'beatgans'
conf.fp16 = True
conf.lr = 1e-4
conf.model_name = ModelName.beatgans_autoenc
conf.net_attn = (16,)
conf.net_beatgans_attn_head = 1
conf.net_beatgans_embed_channels = 512
conf.net_beatgans_resnet_two_cond = True
conf.net_ch_mult = (1, 1, 2, 3, 4)
conf.net_ch = 128
conf.net_enc_channel_mult = (1, 1, 2, 3, 4, 4)
conf.net_enc_pool = 'adaptivenonzero'
conf.img_size = 128
conf.T = 1000
conf.make_model_conf()

autoenc = conf.model_conf.make_model()          # BeatGANsAutoencModel

state  = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
sd_all = state['state_dict']
ema_sd = {k[len('ema_model.'):]: v for k, v in sd_all.items() if k.startswith('ema_model.')}
info   = autoenc.load_state_dict(ema_sd, strict=False)
print(f'EMA keys: {len(ema_sd)} | missing: {len(info.missing_keys)} | unexpected: {len(info.unexpected_keys)}')
print('trained steps:', state.get('global_step'))

ema = autoenc.to(device).eval()
for p in ema.parameters():
    p.requires_grad_(False)

# cond 注入接口自动探测：forward(x, t, cond=?) 或 forward(x, t, y=?)
import inspect
params = inspect.signature(type(ema).forward).parameters
print('forward signature:', inspect.signature(type(ema).forward))
KW = 'cond' if 'cond' in params else ('y' if 'y' in params else None)
assert KW is not None, 'forward 既不接受 cond 也不接受 y，打印 signature 手动确认注入参数名'

def decode(x_t, t, z):
    return ema(x_t, t, **{KW: z})

@torch.no_grad()
def get_z(x):                                    # x in [-1,1]
    return ema.encoder(x)


## 2. 图片池 + 扩散系数

DiffAE 在全部 1000 个 t 上训练，beta 为线性 schedule，与guided-diffusion一致：
betas = linspace(1e-4, 0.02, 1000)。q_sample 手动实现，避免依赖 sampler 内部张量设备。


In [ ]:
from PIL import Image

paths = sorted(glob.glob(os.path.join(IMG_DIR, '*.jpg')))[:VAL_N]
assert len(paths) > 0, f'在 {IMG_DIR} 没找到图片'
imgs = []
for p in paths:
    im = Image.open(p).convert('RGB').resize((128, 128), Image.BILINEAR)
    imgs.append(torch.from_numpy(np.asarray(im)).permute(2, 0, 1).float() / 255.0)
pool01 = torch.stack(imgs).to(device)            # [N,3,128,128], [0,1]，DiffAE 输入直接 *2-1

betas = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64, device=device)
alphas_cumprod = torch.cumprod(1.0 - betas, dim=0)
logsnr_all = torch.log(alphas_cumprod / (1 - alphas_cumprod)).cpu().numpy()

def q_sample(x0, t, noise):
    a = alphas_cumprod[t].float().view(-1, 1, 1, 1)
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

print('pool:', pool01.shape)


## 3. 主循环：遍历全部 1000 个 t（DiffAE 的训练网格就是全量）


In [ ]:
from tqdm import tqdm

assert SAMPLES_PER_T % BATCH_PER_T == 0
rounds = SAMPLES_PER_T // BATCH_PER_T
T = 1000

recs = []
for t_val in tqdm(range(T), desc='scan t'):
    t = torch.full((BATCH_PER_T,), t_val, device=device, dtype=torch.long)
    for _ in range(rounds):
        idx   = torch.randint(0, pool01.shape[0], (BATCH_PER_T,), device=device)
        x0    = pool01[idx] * 2.0 - 1.0
        z     = get_z(x0)
        noise = torch.randn_like(x0)
        x_t   = q_sample(x0, t, noise)

        with torch.no_grad():
            p_c = decode(x_t, t, z)
            p_z = decode(x_t, t, torch.zeros_like(z))
            p_m = decode(x_t, t, z.roll(1, 0))

        l_cond = (p_c - noise).pow(2).flatten(1).mean(1)
        l_zero = (p_z - noise).pow(2).flatten(1).mean(1)
        l_mis  = (p_m - noise).pow(2).flatten(1).mean(1)
        for i in range(BATCH_PER_T):
            recs.append(dict(t=t_val, img=idx[i].item(),
                             l_cond=l_cond[i].item(), l_zero=l_zero[i].item(), l_mis=l_mis[i].item()))

df = pd.DataFrame(recs)
df['d_zero'] = df.l_zero - df.l_cond
df['d_mis']  = df.l_mis  - df.l_cond
df['logsnr'] = logsnr_all[df['t'].values]
df.to_csv(os.path.join(OUT_DIR, 'per_sample.csv'), index=False)
print(df.shape, f'= {T} 个 t x {SAMPLES_PER_T} 个独立采样')
os.makedirs(OUT_DIR, exist_ok=True)


## 4. 聚合 + 图


In [ ]:
g = df.groupby('t').agg(
    l_cond_mean=('l_cond', 'mean'), l_zero_mean=('l_zero', 'mean'), l_mis_mean=('l_mis', 'mean'),
    d_zero_mean=('d_zero', 'mean'), d_zero_med=('d_zero', 'median'), d_zero_std=('d_zero', 'std'),
    d_mis_mean=('d_mis', 'mean'),
    frac_pos=('d_zero', lambda s: (s > 0).mean()),
).reset_index()
g['logsnr'] = logsnr_all[g['t'].values]
g.to_csv(os.path.join(OUT_DIR, 'per_t_agg.csv'), index=False)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, xcol, xlab in [(axes[0], 't', 't'), (axes[1], 'logsnr', 'log SNR')]:
    ax.plot(g[xcol], g.l_cond_mean, label='with z_sem')
    ax.plot(g[xcol], g.l_zero_mean, label='zero z_sem')
    ax.plot(g[xcol], g.l_mis_mean,  label='mismatched z_sem')
    ax.set_xlabel(xlab); ax.set_ylabel('denoising MSE'); ax.legend(fontsize=9)
axes[0].set_title('denoising loss vs t'); axes[1].set_title('denoising loss vs logSNR')
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, 'fig1_loss_curves.png'), dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, xcol, xlab in [(axes[0], 't', 't'), (axes[1], 'logsnr', 'log SNR')]:
    ax.plot(g[xcol], g.d_zero_mean, color='C0', label='D(zero) mean')
    ax.fill_between(g[xcol], g.d_zero_mean - g.d_zero_std, g.d_zero_mean + g.d_zero_std,
                    color='C0', alpha=0.25, label='±1 std (per-sample)')
    ax.plot(g[xcol], g.d_mis_mean, color='C3', label='D(mismatch) mean')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel(xlab); ax.legend(fontsize=8)
i = g.d_zero_mean.values.argmax()
axes[0].set_title(f'argmax of mean D: t={g.t[i]} (logSNR={g.logsnr[i]:.2f})')
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, 'fig2_D_curves.png'), dpi=200); plt.show()


In [ ]:
# ---- 散点 + 混淆检查 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sub = df.iloc[::20]
axes[0].scatter(sub.t, sub.d_zero, s=3, alpha=0.15, color='C0')
axes[0].plot(g.t, g.d_zero_mean, color='k', lw=1.5)
axes[0].axhline(0, color='r', lw=0.5)
axes[0].set_xlabel('t'); axes[0].set_ylabel('D(zero) per sample')
axes[0].set_title('per-sample D (subsampled), mean in black')

df['t_seg'] = pd.cut(df.t, 10)
grp = df.groupby('t_seg', observed=True)
corr = grp.apply(lambda s: s.d_zero.corr(s.l_cond), include_groups=False)
axes[1].plot(grp.t.mean(), corr, marker='o')
axes[1].set_xlabel('t'); axes[1].set_ylabel('corr(D, L_cond)')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('confound check')
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, 'fig3_scatter_confound.png'), dpi=200); plt.show()


## 5. 数值体检 + 汇总


In [ ]:
torch.manual_seed(123)
t_fix = torch.full((8,), 500, device=device, dtype=torch.long)
idx_fix = torch.randint(0, pool01.shape[0], (8,), device=device)
x0_f = pool01[idx_fix] * 2 - 1
z_f  = get_z(x0_f)
noise_f = torch.randn_like(x0_f)
x_t_f = q_sample(x0_f, t_fix, noise_f)
with torch.no_grad():
    p_c = decode(x_t_f, t_fix, z_f)
    p_z = decode(x_t_f, t_fix, torch.zeros_like(z_f))
    p_m = decode(x_t_f, t_fix, z_f.roll(1, 0))

print('1) 预测差异（max abs diff）:')
print('   |pred(zero)-pred(cond)| =', (p_z - p_c).abs().max().item())
print('   |pred(mis) -pred(cond)| =', (p_m - p_c).abs().max().item())
print('2) z_sem per-dim std =', z_f.std(0).mean().item(), ' |mean| =', z_f.mean(0).abs().mean().item())
print('3) 注入相关层权重:')
for n, m in ema.named_modules():
    if any(k in n for k in ['cond_embed', 'time_embed']) and hasattr(m, 'weight'):
        print(f'   {n}: |W| mean={m.weight.abs().mean().item():.5f}')
_k0 = next(iter(ema_sd))
print('4) EMA vs 在线权重(前3个):')
print('   ema :', ema_sd[_k0].flatten()[:3].tolist())
print('   raw :', sd_all['model.' + _k0].flatten()[:3].tolist())


In [ ]:
print('=== 汇总 ===')
print(f"L_cond 全体均值: {df.l_cond.mean():.4f}")
print(f"D(zero)>0 比例: {(df.d_zero > 0).mean():.3f}")
print(f"D 量级: mean|d_zero|={df.d_zero.abs().mean():.6f}  vs  mean L_cond={df.l_cond.mean():.4f}")
print(f"D(zero) 峰值: t={g.t[g.d_zero_mean.values.argmax()]}  "
      f"D={g.d_zero_mean.max():.6f}   D(mis) 峰值: t={g.t[g.d_mis_mean.values.argmax()]}  "
      f"D={g.d_mis_mean.max():.6f}")
print()
print('判读:')
print(' 健康参照的预期: D 稳定为正、幅度 >= L_cond 的 1% 量级、')
print('                 且 D(mismatch) 与 D(zero) 在相同 t 一起显著为正 <- 信息被使用的签名')
print(' 与你们模型(baseline_diffae_ade20k@80k)对照:')
print('   你们: D>0比例0.62, mean|d_zero|=1e-6, D(mis)≈0, z_sem std=5e-5  -> 死通路')
